# 03T — Analysis sandbox  ·  line T (testing)

Where a new analysis gets tried before it earns a place in `03A`/`03B`.

**The rule this line exists to enforce:** `src/` holds *general* functions. Something
hardcoded to one region, one dataset or one figure — called once to make one plot — stays in
a notebook. This is that notebook. When something here starts being useful for every region,
it moves into `src/` and gains a step number in `03A` and `03B`.

Seeded below with the questions that are actually still open. Each has the setup written out
so it can be run, and a note on what would settle it.

In [ ]:
import pathlib
import sys

project_root = pathlib.Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root.resolve()))

import matplotlib.pyplot as plt
import numpy as np

from src import analysis, config, oscillation, spectra
from src.loaders import load_ds0n_region, load_noaa_region

SAVE = False

---

## 1. Is the slow magnetogram trend just the changing viewing geometry?

`notes.md`: *"el h mu es una parabola podria ser la razon por la que los datos de
magnetograma parecen una parabola"*.

The magnetogram's mean over the spot drifts across a multi-day window in a shape close to a
parabola. So does μ = cos(heliocentric angle), because the spot rotates from one side of the
disk toward the other. If the two track each other, the trend is foreshortening rather than
anything solar — and `add_mag_residuals`, which subtracts a parabola, is removing geometry
rather than a systematic.

`analysis.plot_mu_vs_trend` puts the two normalised curves on one axis. Normalised, so it can
say *same shape* or *opposite shape* and nothing about size.

**What would settle it:** the residual after subtracting a μ-driven model, rather than an
arbitrary parabola. If the field really goes as `B_los = B · μ`, dividing by μ is the
physical correction and the leftover is the real variation.

In [ ]:
ds_dir = config.DS0N_RAW_DIR / 'DS01'
params = config.params_for(ds_dir, line='B')
data = load_ds0n_region(ds_dir, **config.loader_kwargs(params))
metrics = analysis.add_mag_residuals(data, analysis.compute_metrics(data))

cube_mu = analysis.mu_cube_ds0n(ds_dir)
mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data)
analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag')

In [ ]:
# The physical version: divide the LOS field by mu before averaging, so the geometry is
# taken out rather than fitted away. If the parabola in mean_mag_umb is foreshortening, this
# should flatten it; if it survives, the trend is something else.
umbra = data['umbra']
mu_safe = np.where(cube_mu > 0.05, cube_mu, np.nan)
b_corrected = data['cube_mag'] / mu_safe

from src.utilities import mean_series
mean_b_los = mean_series(data['cube_mag'], umbra)
mean_b_mu  = mean_series(b_corrected, umbra)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(data['time_h'], mean_b_los, lw=0.9, label='B_los')
ax.plot(data['time_h'], mean_b_mu, lw=0.9, label='B_los / mu')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Mean umbral field (G)')
ax.set_title('Does dividing by mu flatten the trend?')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for name, series in (('B_los', mean_b_los), ('B_los / mu', mean_b_mu)):
    curvature = np.polyfit(data['time_h'], np.nan_to_num(series, nan=np.nanmean(series)), 2)[0]
    print(f'{name:12s} quadratic coefficient: {curvature:+.4e} G/h^2')

---

## 2. How wide should the notch be?

`spectra.notch_filter` zeroes every bin within `width_mhz` of the target. The prototype had
two implementations that disagreed — one zeroed a band, the other only the nearest bin — and
the merged version uses the band.

The width is not free: too narrow and leakage leaves the tone behind, too wide and it takes
real neighbouring signal with it, and either way zeroing a band rings in the time domain.

**What would settle it:** the sweep below. Pick the smallest width that drives the residual
at the target period into the noise, and check on the time-domain panel that it has not
started eating the rest.

In [ ]:
series = metrics['mean_dop_umb']
cadence_s = data['cadence_s']
period_min = 1440.0
target_mhz = 1e3 / (period_min * 60)

fig, ax = plt.subplots(figsize=(11, 5))
print(f'{"width (mHz)":>12} {"bins":>6} {"residual at target":>20} {"std of the rest":>18}')
for width in (0.00005, 0.0001, 0.0002, 0.0005, 0.001, 0.002):
    filtered, n_bins = spectra.notch_filter(series, cadence_s, period_min, width_mhz=width)
    freq, amp = spectra.fft_spectrum(filtered, cadence_s, window='hann', scale='amplitude')
    near = np.abs(freq - target_mhz) < 0.003
    print(f'{width:12.5f} {n_bins:6d} {amp[near].max():20.2f} {np.nanstd(filtered):18.2f}')
    ax.plot(data['time_h'][:len(filtered)], filtered, lw=0.8, label=f'{width:g} mHz')

ax.plot(data['time_h'], spectra.interpolate_gaps(series), lw=0.8, color='0.6',
        label='unfiltered')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Mean umbral velocity (m/s)')
ax.set_title('Notch width sweep at 1440 min')
ax.legend(fontsize=8, ncol=3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 3. Do the umbra and the quiet sun really move in antiphase?

`notes.md`: *"la oscilacion del magnetrograma residual normalizado resulta ser
convientemente opuesta al del doplergrama"*, and the working conclusion that the cleaning
subtracted more than it should have.

The `04` fits say the umbral fraction of the amplitude is close to 1 for every NOAA region,
i.e. the umbra's motion largely survives subtracting the quiet sun. But antiphase between the
magnetogram residual and the Doppler signal is a different statement and has not been
measured — only seen.

**What would settle it:** a phase, with an uncertainty. `fit_diurnal` returns `t_max`, so
the phase difference between two series is available directly; the cross-correlation below
is the model-free version of the same question.

In [ ]:
time_h = data['time_h']
v = metrics['mean_dop_umb'] - metrics['mean_dop_quiet']
b = metrics['mean_mag_umb_residual']

fit_v = oscillation.fit_diurnal(time_h, v, period_h=24.0, label='umbral velocity')
fit_b = oscillation.fit_diurnal(time_h, b, period_h=24.0, label='umbral B residual')
lag_h = (fit_b['t_max'] - fit_v['t_max']) % 24.0
print(f'v   peaks at t_max = {fit_v["t_max"]:6.2f} h,  A = {fit_v["amplitude"]:8.2f} '
      f'± {fit_v["sigma_amplitude"]:.2f} m/s')
print(f'B   peaks at t_max = {fit_b["t_max"]:6.2f} h,  A = {fit_b["amplitude"]:8.2f} '
      f'± {fit_b["sigma_amplitude"]:.2f} G')
print(f'lag = {lag_h:.2f} h of 24 h = {360 * lag_h / 24:.0f} deg   '
      f'(antiphase would be ~12 h / 180 deg)')

# Model-free: cross-correlate the detrended series and read off the lag at the peak.
vv = spectra.detrend_poly(time_h, v, 1)
bb = spectra.detrend_poly(time_h, b, 1)
good = np.isfinite(vv) & np.isfinite(bb)
vv, bb = vv[good] - vv[good].mean(), bb[good] - bb[good].mean()
corr = np.correlate(vv / np.std(vv), bb / np.std(bb), mode='full') / len(vv)
lags_h = (np.arange(-len(vv) + 1, len(vv))) * data['cadence_s'] / 3600

window = np.abs(lags_h) <= 36
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(lags_h[window], corr[window], lw=1.0)
ax.axvline(0, color='0.5', lw=0.8)
ax.set_xlabel('lag (h)   — positive: B lags v')
ax.set_ylabel('normalised cross-correlation')
ax.set_title('Umbral velocity vs magnetogram residual')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'peak correlation {corr[window].max():+.3f} at '
      f'{lags_h[window][np.argmax(corr[window])]:+.2f} h')

---

## 4. The reconstructed dopplergram does not match the shipped one

`01T_coefficient_reconstruction.ipynb` inverts HMI's cubic calibration to rebuild DS01's
dopplergram from the raw cube. The result correlates at about zero with the shipped
`cube_dopplergram_corrected.fits`.

The likeliest reading is that the two correct different things: the cubic describes the
tuning nonlinearity, while the shipped file looks like it carries an orbital or
gravitational-redshift correction instead.

**What would settle it:** compare the *difference* `shipped − raw` against the known
analytic terms in `src/doppler_calibration.py` (`sdo_los_velocity_keywords`,
`large_scale_flow_los`, `convective_blueshift`, `GRAVITATIONAL_REDSHIFT`). If the difference
is dominated by one of them, the question is answered without any reconstruction at all.

In [ ]:
# Left as the next thing to run, not as a result. See 01T for the reconstruction itself.
from astropy.io import fits

ds01 = config.DS0N_RAW_DIR / 'DS01'
with fits.open(ds01 / 'cube_dopplergram.fits') as h:
    raw = h[0].data.astype(np.float32)
with fits.open(ds01 / 'cube_dopplergram_corrected.fits') as h:
    shipped = h[0].data.astype(np.float32)

difference = shipped - raw
per_frame = np.array([np.nanmean(f, dtype=np.float64) for f in difference])
print(f'shipped - raw, per-frame mean: {np.nanmin(per_frame):.1f} .. '
      f'{np.nanmax(per_frame):.1f} m/s  (span {np.ptp(per_frame[np.isfinite(per_frame)]):.1f})')
print(f'gravitational redshift alone would be a constant 636.03 m/s')

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(per_frame, lw=0.9)
ax.set_xlabel('frame')
ax.set_ylabel('mean(shipped - raw)  (m/s)')
ax.set_title('What did the delivered correction actually remove?')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 5. Subtract the quiet-sun *cube*, not its mean

`notes.md`: *"se podria restar el cubo_Qsun con el de las oscilaciones forma de que se pueda
identificar las velocidades relativas"*.

Everything so far subtracts a per-frame quiet-sun *scalar*. That removes the spatially
uniform part and leaves the line-of-sight gradient across the box, which the umbra samples
off-centre — so a residual drift survives that has nothing to do with the spot.

`processing.remove_quiet_sun_plane` already does the plane version for the magnetogram, and
`config.PROCESSING['residual_plane_fit']` turns it on for the dopplergram too. It is off by
default because it also throws away the absolute scale the physical corrections established.

**What would settle it:** process one region both ways and compare the fitted amplitudes. If
they agree, the gradient does not matter at 24 h and the question is closed.

In [ ]:
# Scaffolding for that comparison. Uncomment to run — it re-corrects a whole region, ~90 s.
#
# from src import processing
# region_dir = config.RAW_DIR / 'NOAA_11363_2011-12-06'
# settings = dict(config.PROCESSING)
# settings['residual_plane_fit'] = True
# result = processing.process_region(region_dir, **settings)
# ... then build regions from result['cubes'] and fit, against the same region from
#     data/processed/ which was made with residual_plane_fit = False.
print('see the comment above — this one changes what 02A writes, so it needs its own output '
      'directory before it is run in anger')